# Lab 07 — Fine-tuning: full vs LoRA, memory and loss curves

**Goal:** run your proposal's actual workload at toy scale: fine-tune GPT-2 on WikiText, first **full** (all 124M params, Adam) then **LoRA**, and measure the two things your evaluation always reports — **peak VRAM** and the **loss curve**.

In [ ]:
!pip -q install transformers peft datasets matplotlib
import torch, matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
device = "cuda"
tok = AutoTokenizer.from_pretrained("gpt2"); tok.pad_token = tok.eos_token

## 1. Data — tokenize and pack into fixed blocks

In [ ]:
raw = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:2%]")
text = "\n\n".join(t for t in raw["text"] if t.strip())
enc = tok(text, return_tensors="pt").input_ids[0]
SEQ = 128
n = enc.numel() // SEQ
data = enc[:n*SEQ].view(n, SEQ)
print("training blocks:", data.shape)

## 2. A minimal, honest training loop (used for both runs)

In [ ]:
def train(model, steps=150, bs=8, lr=5e-5, tag=""):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    torch.cuda.reset_peak_memory_stats()
    losses = []
    model.train()
    for step in range(steps):
        idx = torch.randint(0, data.size(0), (bs,))
        batch = data[idx].to(device)
        loss = model(batch, labels=batch).loss
        loss.backward(); opt.step(); opt.zero_grad()
        losses.append(loss.item())
        if step % 25 == 0: print(f"{tag} step {step:3d}  loss {loss.item():.3f}")
    peak = torch.cuda.max_memory_allocated()/1e9
    print(f"{tag} peak VRAM: {peak:.2f} GB")
    return losses, peak

## 3. Run A — full fine-tuning (every parameter gets grads + Adam states)

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
print("trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6, "M")
full_losses, full_peak = train(model, tag="[full]")
del model; torch.cuda.empty_cache()

## 4. Run B — LoRA (frozen base, tiny adapters)

In [ ]:
from peft import LoraConfig, get_peft_model
base = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
lcfg = LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], task_type="CAUSAL_LM")
lora_model = get_peft_model(base, lcfg)
lora_model.print_trainable_parameters()
lora_losses, lora_peak = train(lora_model, tag="[lora]")

## 5. Compare — the two plots your final evaluation is made of

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,3.5))
ax[0].plot(full_losses, label="full"); ax[0].plot(lora_losses, label="LoRA")
ax[0].set_title("Loss curves ('at loss parity' = these must overlap for your system)")
ax[0].set_xlabel("step"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].bar(["full", "LoRA"], [full_peak, lora_peak])
ax[1].set_title("Peak VRAM (GB) — the 16-bytes/param story, live")
plt.tight_layout(); plt.show()

Notice: LoRA slashes memory, but if you re-ran Lab 03's hooks here, the **boundary activations are identical in size** — LoRA rescues VRAM without shrinking your communication problem by one byte. That's why the proposal remains fully motivated even for the parameter-efficient workload.

## Exercises
1. Sweep LoRA rank r in [2, 8, 32]: trainable params, memory, final loss.
2. Add `target_modules=["c_attn", "c_fc"]` — better loss for how much more memory?
3. Generate from the model before vs after fine-tuning on the same prompt — see the behavior change.
4. Combine with Lab 05: LoRA model, staged, quantized wire — your first end-to-end miniature of the whole system.